## Script

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import yaml
from pathlib import Path

# Configuración central
CONFIG = {
    'seed': 42,
    'start_date': '2024-03-01',
    'end_date': '2026-03-31',
    'countries': ['Colombia', 'Mexico', 'Peru', 'Chile', 'Argentina'],
    'table_volumes': {
        'TB_CLIENTES_CORE': 10000,
        'TB_PRODUCTOS_CAT': 50,
        'TB_MOV_FINANCIEROS': 500000,
        'TB_OBLIGACIONES': 30000,
        'TB_SUCURSALES_RED': 200,
        'TB_COMISIONES_LOG': 80000
    },
    'null_percentage': 0.05
}

np.random.seed(CONFIG['seed'])

# ============= TB_CLIENTES_CORE =============
def generate_clientes_core():
    n = CONFIG['table_volumes']['TB_CLIENTES_CORE']
    start = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
    
    data = {
        'id_cli': np.arange(1, n + 1),
        'nomb_cli': np.random.choice(['Juan', 'Maria', 'Carlos', 'Ana', 'Pedro', 'Laura', 'Miguel'], n),
        'apell_cli': np.random.choice(['Gonzalez', 'Rodriguez', 'Martinez', 'Lopez', 'Hernandez', 'Garcia'], n),
        'tip_doc': np.random.choice(['CC', 'CE', 'PEP'], n),
        'num_doc': [f'{1000000000 + i:010d}' for i in range(n)],
        'fec_nac': [start - timedelta(days=np.random.randint(7300, 27375)) for _ in range(n)],  # 20-75 años
        'fec_alta': [start + timedelta(days=np.random.randint(0, 730)) for _ in range(n)],  # Últimos 2 años
        'cod_segmento': np.random.choice(['B', 'S', 'P', 'E'], n),  # Básico, Estándar, Premium, Elite
        'score_buro': np.random.randint(300, 900, n),
        'ciudad_res': np.random.choice(['Bogotá', 'CDMX', 'Lima', 'Santiago', 'Buenos Aires', 'Medellín', 'Guadalajara'], n),
        'depto_res': np.random.choice(['Cundinamarca', 'Jalisco', 'Lima', 'Metropolitana', 'Buenos Aires'], n),
        'estado_cli': np.random.choice(['Activo', 'Inactivo', 'Bloqueado'], n, p=[0.85, 0.10, 0.05]),
        'canal_adquis': np.random.choice(['APP_MOBILE', 'WEB', 'CORRESPONSAL'], n)
    }
    
    df = pd.DataFrame(data)
    
    # Insertar ~5% nulos en campos no críticos
    null_cols = ['depto_res', 'canal_adquis']
    for col in null_cols:
        null_idx = np.random.choice(df.index, size=int(len(df) * CONFIG['null_percentage']), replace=False)
        df.loc[null_idx, col] = None
    
    return df

# ============= TB_PRODUCTOS_CAT =============
def generate_productos_cat():
    productos = [
        ('P001', 'Crédito Libre Inversión', 'CREDITO', 0.25, 60, 500000, 0.03, 'Activo'),
        ('P002', 'Crédito Rotativo', 'CREDITO', 0.22, 36, 100000, 0.02, 'Activo'),
        ('P003', 'Tarjeta Digital', 'CREDITO', 0.28, 12, 50000, 0.04, 'Activo'),
        ('P004', 'Ahorro Básico', 'AHORRO', 0.01, 0, 0, 0.0, 'Activo'),
        ('P005', 'Ahorro Plazo Fijo', 'AHORRO', 0.04, 60, 0, 0.0, 'Activo'),
        ('P006', 'Pagos PSE', 'TRANSACCIONAL', 0.0, 0, 0, 0.005, 'Activo'),
        ('P007', 'Transferencias ACH', 'TRANSACCIONAL', 0.0, 0, 0, 0.003, 'Activo'),
        ('P008', 'Corresponsalía', 'TRANSACCIONAL', 0.0, 0, 0, 0.01, 'Activo'),
    ]
    
    # Expandir para llegar a 50 productos
    expanded = []
    for i in range(CONFIG['table_volumes']['TB_PRODUCTOS_CAT']):
        base = productos[i % len(productos)]
        expanded.append((
            f'P{i+1:03d}',
            base[1] + (f' V{i//len(productos)}' if i >= len(productos) else ''),
            base[2],
            base[3],
            base[4],
            base[5],
            base[6],
            base[7]
        ))
    
    df = pd.DataFrame(expanded, columns=[
        'cod_prod', 'desc_prod', 'tip_prod', 'tasa_ea', 'plazo_max_meses',
        'cuota_min', 'comision_admin', 'estado_prod'
    ])
    
    return df

# ============= TB_MOV_FINANCIEROS =============
def generate_mov_financieros():
    n = CONFIG['table_volumes']['TB_MOV_FINANCIEROS']
    start = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
    end = datetime.strptime(CONFIG['end_date'], '%Y-%m-%d')
    
    n_clientes = CONFIG['table_volumes']['TB_CLIENTES_CORE']
    n_productos = CONFIG['table_volumes']['TB_PRODUCTOS_CAT']
    
    data = {
        'id_mov': np.arange(1, n + 1),
        'id_cli': np.random.randint(1, n_clientes + 1, n),
        'cod_prod': np.random.choice([f'P{i+1:03d}' for i in range(n_productos)], n),
        'num_cuenta': [f'{100000000 + i:010d}' for i in range(n)],
        'fec_mov': [start + timedelta(days=np.random.randint(0, (end-start).days)) for _ in range(n)],
        'hra_mov': [f'{np.random.randint(0, 24):02d}:{np.random.randint(0, 60):02d}:{np.random.randint(0, 60):02d}' for _ in range(n)],
        'vr_mov': np.random.exponential(scale=500000, size=n).astype(int),  # Distribución realista
        'tip_mov': np.random.choice(['Deposito', 'Retiro', 'Transferencia', 'Pago'], n),
        'cod_canal': np.random.choice(['APP', 'WEB', 'ATM', 'CORRESPONSAL'], n),
        'cod_ciudad': np.random.choice(['001', '002', '003', '004', '005'], n),
        'cod_estado_mov': np.random.choice(['Exitosa', 'Fallida', 'Pendiente'], n, p=[0.95, 0.03, 0.02]),
        'id_dispositivo': [f'DEV_{i%1000:04d}' for i in range(n)]
    }
    
    df = pd.DataFrame(data)
    
    # Anomalías intencionales
    # 1. Transacciones duplicadas
    dup_indices = np.random.choice(df.index, size=100, replace=False)
    df = pd.concat([df, df.iloc[dup_indices].reset_index(drop=True)], ignore_index=True)
    
    # 2. Fechas fuera de rango
    df.loc[df.sample(frac=0.001).index, 'fec_mov'] = datetime(1900, 1, 1)
    
    return df

# ============= TB_OBLIGACIONES =============
def generate_obligaciones():
    n = CONFIG['table_volumes']['TB_OBLIGACIONES']
    n_clientes = CONFIG['table_volumes']['TB_CLIENTES_CORE']
    n_productos = CONFIG['table_volumes']['TB_PRODUCTOS_CAT']
    start = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
    
    data = {
        'id_oblig': np.arange(1, n + 1),
        'id_cli': np.random.randint(1, n_clientes + 1, n),
        'cod_prod': np.random.choice([f'P{i+1:03d}' for i in range(min(3, n_productos))], n),  # Solo créditos
        'vr_aprobado': np.random.exponential(scale=10000000, size=n).astype(int),
        'vr_desembolsado': np.random.exponential(scale=8000000, size=n).astype(int),
        'sdo_capital': np.random.exponential(scale=7000000, size=n).astype(int),
        'vr_cuota': np.random.exponential(scale=500000, size=n).astype(int),
        'fec_desembolso': [start + timedelta(days=np.random.randint(-365, 0)) for _ in range(n)],
        'fec_venc': [start + timedelta(days=np.random.randint(0, 1800)) for _ in range(n)],
        'dias_mora_act': np.random.choice(
            [0, 5, 15, 45, 75, 120],  # Distribución realista
            n,
            p=[0.70, 0.15, 0.08, 0.04, 0.02, 0.01]
        ),
        'num_cuotas_pend': np.random.randint(0, 61, n),
        'calif_riesgo': np.random.choice(['A', 'B', 'C', 'D', 'E'], n, p=[0.60, 0.20, 0.10, 0.07, 0.03])
    }
    
    df = pd.DataFrame(data)
    
    # Anomalía 3: Inconsistencia lógica - saldo de capital mayor que desembolsado
    # (Lógicamente inconsistente: no se puede deber más de lo que se desembolsó)
    inconsistent_idx = np.random.choice(df.index, size=int(len(df) * 0.02), replace=False)
    df.loc[inconsistent_idx, 'sdo_capital'] = (df.loc[inconsistent_idx, 'vr_desembolsado'] * 1.5).round().astype(int)    
    return df

# ============= TB_SUCURSALES_RED =============
def generate_sucursales_red():
    n = CONFIG['table_volumes']['TB_SUCURSALES_RED']
    
    ciudades = {
        'Bogotá': (-74.0060, 4.7110),
        'CDMX': (-99.1332, 19.4326),
        'Lima': (-77.0282, -12.0464),
        'Santiago': (-70.6673, -33.4489),
        'Buenos Aires': (-58.3816, -34.6037),
    }
    
    data = {
        'cod_suc': [f'SUC_{i+1:04d}' for i in range(n)],
        'nom_suc': [f'Sucursal {i+1}' for i in range(n)],
        'tip_punto': np.random.choice(['Oficina', 'Corresponsal', 'ATM'], n, p=[0.3, 0.5, 0.2]),
        'ciudad': np.random.choice(list(ciudades.keys()), n),
        'depto': np.random.choice(['Cundinamarca', 'Jalisco', 'Lima', 'Metropolitana', 'Buenos Aires'], n),
        'latitud': [ciudades[ciudad][1] + np.random.uniform(-0.5, 0.5) for ciudad in np.random.choice(list(ciudades.keys()), n)],
        'longitud': [ciudades[ciudad][0] + np.random.uniform(-0.5, 0.5) for ciudad in np.random.choice(list(ciudades.keys()), n)],
        'activo': np.random.choice([True, False], n, p=[0.95, 0.05])
    }
    
    df = pd.DataFrame(data)
    return df

# ============= TB_COMISIONES_LOG =============
def generate_comisiones_log():
    n = CONFIG['table_volumes']['TB_COMISIONES_LOG']
    n_clientes = CONFIG['table_volumes']['TB_CLIENTES_CORE']
    n_productos = CONFIG['table_volumes']['TB_PRODUCTOS_CAT']
    start = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
    end = datetime.strptime(CONFIG['end_date'], '%Y-%m-%d')
    
    data = {
        'id_comision': np.arange(1, n + 1),
        'id_cli': np.random.randint(1, n_clientes + 1, n),
        'cod_prod': np.random.choice([f'P{i+1:03d}' for i in range(n_productos)], n),
        'fec_cobro': [start + timedelta(days=np.random.randint(0, (end-start).days)) for _ in range(n)],
        'vr_comision': np.random.exponential(scale=50000, size=n).astype(int),
        'tip_comision': np.random.choice(['Mantenimiento', 'Transaccional', 'Mora'], n, p=[0.5, 0.3, 0.2]),
        'estado_cobro': np.random.choice(['Cobrada', 'Pendiente', 'Rechazada'], n, p=[0.90, 0.07, 0.03])
    }
    df = pd.DataFrame(data)
    return df


StatementMeta(, 8f519166-ce94-45db-bede-abbce09ab006, 4, Finished, Available, Finished, False)

In [2]:
def generate_all_data():
    print("Generando datos sintéticos: ")
    
    tables = { 
        'TB_CLIENTES_CORE': generate_clientes_core(),
        'TB_PRODUCTOS_CAT': generate_productos_cat(),
        'TB_MOV_FINANCIEROS': generate_mov_financieros(),
        'TB_OBLIGACIONES': generate_obligaciones(),
        'TB_SUCURSALES_RED': generate_sucursales_red(),
        'TB_COMISIONES_LOG': generate_comisiones_log(),
    }
    
    base_parquet = "Files/raw_parquet"
    base_csv = "Files/raw_csv"
    base_json = "Files/raw_json"
    
    for table_name, df in tables.items():
        print(f"Procesando {table_name}...")
        
        spark_df = spark.createDataFrame(df)
        
        # =========================
        # 🥉 PARQUET
        # =========================
        spark_df.write.mode("overwrite").parquet(f"{base_parquet}/{table_name}")
        
        # =========================
        # 📄 CSV (Spark)
        # =========================
        spark_df.write.mode("overwrite").option("header", True).csv(f"{base_csv}/{table_name}")
        
        # =========================
        # 📄 JSON (Spark)
        # =========================
        spark_df.write.mode("overwrite").json(f"{base_json}/{table_name}")
        
        # =========================
        # 🟢 TABLA SQL
        # =========================
        spark_df.write.mode("overwrite").saveAsTable(f"dbo.{table_name}")
        
        print(f"✅ {table_name}: {spark_df.count()} registros")
    
    print("✅ Datos generados en parquet, csv, json y tablas")
    return tables


if __name__ == '__main__':
    generate_all_data()

StatementMeta(, 8f519166-ce94-45db-bede-abbce09ab006, 5, Finished, Available, Finished, False)

Generando datos sintéticos: 


Procesando TB_CLIENTES_CORE...


✅ TB_CLIENTES_CORE: 10000 registros
Procesando TB_PRODUCTOS_CAT...


✅ TB_PRODUCTOS_CAT: 50 registros
Procesando TB_MOV_FINANCIEROS...


✅ TB_MOV_FINANCIEROS: 500100 registros
Procesando TB_OBLIGACIONES...


✅ TB_OBLIGACIONES: 30000 registros
Procesando TB_SUCURSALES_RED...


✅ TB_SUCURSALES_RED: 200 registros
Procesando TB_COMISIONES_LOG...


✅ TB_COMISIONES_LOG: 80000 registros
✅ Datos generados en parquet, csv, json y tablas
